# AUDIT-GRADE eval: base vs fine-tuned, fully defensible for official review
- 512-token budget for BOTH models (no truncation-unfairness)
- Saves EVERY raw output (auditable)
- Reports parse-fail PER MODEL (proves base was not cut off)
- Identical prompt + Draft-2020-12 validator + greedy for both
- JSONSchemaBench TEST split (decontaminated from training)

In [ ]:
%pip install -q "transformers>=4.55,<5" accelerate "jsonschema[format]" datasets

In [ ]:
import os, json, re, torch, random
os.environ["HF_TOKEN"]=os.environ.get("HF_TOKEN", "")
BASE="Qwen/Qwen3-1.7B"; FT="ujwal00/qwen3-1.7b-json"
SUBSETS=["Github_trivial","Github_easy","Github_medium","Github_hard","Glaiveai2K","Kubernetes","Snowplow","JsonSchemaStore"]
PER=40; MAXNEW=512
from jsonschema import Draft202012Validator
FMT=Draft202012Validator.FORMAT_CHECKER
SYSTEM=("You are a precise JSON generator. Given a JSON Schema, output ONE JSON value "
  "that strictly validates against it. Output ONLY the JSON value - no prose, no "
  "markdown fences, no comments. Include every required property, use only the "
  "properties and types the schema allows, and respect enums, formats, and nesting.")
USER_T="Generate a JSON value that validates against this JSON Schema:\n\n{schema}\n\nReturn only the JSON value."
def render(s):
    if isinstance(s,str): s=json.loads(s)
    return [{"role":"system","content":SYSTEM},{"role":"user","content":USER_T.format(schema=json.dumps(s,sort_keys=True,ensure_ascii=False))}]
_F=re.compile(r"^```(?:json)?\s*|\s*```$",re.M)
def extract_json(text):
    t=_F.sub("",text).strip()
    try: return json.loads(t),True
    except Exception: pass
    st=None
    for i,c in enumerate(t):
        if c in "{[": st=i;break
    if st is None: return None,False
    op=t[st];cl="}" if op=="{" else "]";d=0;ins=False;esc=False
    for j in range(st,len(t)):
        c=t[j]
        if ins:
            if esc:esc=False
            elif c=='\\':esc=True
            elif c=='"':ins=False
        else:
            if c=='"':ins=True
            elif c==op:d+=1
            elif c==cl:
                d-=1
                if d==0:
                    try:return json.loads(t[st:j+1]),True
                    except Exception:return None,False
    return None,False
def valid(inst,s):
    if isinstance(s,str): s=json.loads(s)
    try: return len(list(Draft202012Validator(s,format_checker=FMT).iter_errors(inst)))==0
    except Exception: return False
def nested_of(s):
    x=json.dumps(json.loads(s) if isinstance(s,str) else s)
    return '$ref' in x or '"$defs"' in x or '"definitions"' in x

In [ ]:
from datasets import load_dataset
random.seed(1)
items=[]
for sub in SUBSETS:
    try: ds=load_dataset("epfl-dlab/JSONSchemaBench",sub,split="test")
    except Exception:
        try: ds=load_dataset("epfl-dlab/JSONSchemaBench",sub,split="val")
        except Exception: continue
    for i in random.sample(range(len(ds)),min(PER,len(ds))):
        s=ds[i].get('json_schema') or ds[i].get('schema')
        try: sc=json.loads(s) if isinstance(s,str) else s
        except Exception: continue
        if isinstance(sc,dict): items.append({'sub':sub,'schema':json.dumps(sc),'nested':nested_of(sc)})
print('items',len(items),'nested',sum(x['nested'] for x in items))

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
def run(model_id):
    tok=AutoTokenizer.from_pretrained(model_id,token=os.environ["HF_TOKEN"])
    if tok.pad_token is None: tok.pad_token=tok.eos_token
    tok.padding_side="left"
    m=AutoModelForCausalLM.from_pretrained(model_id,dtype=torch.float16,device_map="auto",token=os.environ["HF_TOKEN"]); m.eval()
    prompts=[tok.apply_chat_template(render(x['schema']),tokenize=False,add_generation_prompt=True,enable_thinking=False) for x in items]
    outs=[]
    for i in range(0,len(prompts),24):
        enc=tok(prompts[i:i+24],return_tensors="pt",padding=True,truncation=True,max_length=2048).to(m.device)
        with torch.no_grad():
            g=m.generate(**enc,max_new_tokens=MAXNEW,do_sample=False,pad_token_id=tok.eos_token_id)
        outs+=tok.batch_decode(g[:,enc.input_ids.shape[1]:],skip_special_tokens=True)
    recs=[]
    for x,t in zip(items,outs):
        obj,p=extract_json(t)
        recs.append({'sub':x['sub'],'nested':x['nested'],'ok':bool(p and valid(obj,x['schema'])),'parsed':p,'out':t[:600],'truncated':len(tok(t)['input_ids'])>=MAXNEW-2})
    del m; torch.cuda.empty_cache()
    return recs
base=run(BASE); ft=run(FT)

In [ ]:
from collections import defaultdict
from math import comb
def summ(recs,mask=lambda x:True):
    r=[x for x in recs if mask(x)]
    return sum(x['ok'] for x in r)/max(len(r),1)
print('=== PER-MODEL HEALTH (proves fairness) ===')
for nm,R in [('BASE',base),('FT',ft)]:
    pf=sum(1 for x in R if not x['parsed'])/len(R); tr=sum(x['truncated'] for x in R)/len(R)
    print(f"{nm}: overall {summ(R):.1%} | parse_fail {pf:.1%} | truncated_at_{MAXNEW}tok {tr:.1%}")
print('\n=== PER-DATASET ===')
for s in SUBSETS:
    b=summ(base,lambda x:x['sub']==s); f=summ(ft,lambda x:x['sub']==s)
    if any(x['sub']==s for x in base): print(f"{s:16s} base {b:.0%} -> ft {f:.0%}  ({f-b:+.0%})")
print(f"\nFLAT   base {summ(base,lambda x:not x['nested']):.0%} -> ft {summ(ft,lambda x:not x['nested']):.0%}")
print(f"NESTED base {summ(base,lambda x:x['nested']):.0%} -> ft {summ(ft,lambda x:x['nested']):.0%}")
print(f"OVERALL base {summ(base):.0%} -> ft {summ(ft):.0%}")
b=sum(1 for x,y in zip(base,ft) if not x['ok'] and y['ok']); c=sum(1 for x,y in zip(base,ft) if x['ok'] and not y['ok'])
n=b+c; p=min(1.0,sum(comb(n,i) for i in range(0,min(b,c)+1))/2**n*2) if n else 1.0
print(f"\nMcNemar: fixed {b} broke {c} net {b-c} p={p:.2e}")
json.dump({'items':len(items),'base':base,'ft':ft,
  'base_overall':summ(base),'ft_overall':summ(ft),
  'base_parsefail':sum(1 for x in base if not x['parsed'])/len(base),
  'ft_parsefail':sum(1 for x in ft if not x['parsed'])/len(ft),
  'base_trunc':sum(x['truncated'] for x in base)/len(base),
  'mcnemar':{'fixed':b,'broke':c,'p':p}},open('/kaggle/working/audit_eval.json','w'),indent=1)
print('\nsaved audit_eval.json (raw outputs included for audit)')